In [1]:
# Import packages
import json
from huggingface_hub import login
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, pipeline
import transformers
import random
import torch
import time
import re
from tqdm import tqdm
import pandas as pd

# Set GPUs
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3,5"

print("Available GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# Set seeds
random.seed(0)
torch.manual_seed(0)


/home/sswee/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Available GPUs: 2
GPU 0: NVIDIA A100-SXM4-40GB
GPU 1: NVIDIA A100-SXM4-40GB


In [2]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load BioBERT tokenizer and model
model_name = "dmis-lab/biobert-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [3]:
# Get clinical note
df = pd.read_csv("../Data/subject-info-cleaned-with-prompts-and-notes-combined-deaths_Llama3B.csv")

In [6]:
# Tokenize clinical note
inputs = tokenizer(report, return_tensors = "pt", truncation = True, padding = True)

In [7]:
# Get model output
# Forward pass through the model to get embeddings
with torch.no_grad():
    outputs = model(**inputs)

# The last hidden states are the embeddings
embeddings = outputs.last_hidden_state  # Shape: [batch_size, sequence_length, hidden_size]


In [8]:
# Option 1: Get embedding for [CLS] token (first token)
cls_embedding = embeddings[:, 0, :]
cls_embedding

tensor([[-1.5974e-01, -4.5796e-03, -4.1298e-01, -6.1571e-03, -7.0118e-02,
         -8.6129e-02,  1.3160e-01, -1.0302e-01,  3.1286e-02,  1.5537e-01,
          6.6901e-02,  3.2824e-01, -6.2212e-01, -2.7106e-01, -4.5849e-01,
         -5.3836e-03,  1.6038e-01, -2.7869e-01,  3.1675e-01, -1.7320e-03,
         -2.0885e-01, -1.3863e-01, -1.3199e-01, -1.2180e-01, -1.0147e-02,
          6.5699e-02,  5.3449e-01,  6.9509e-02, -4.8038e-01,  6.7263e-01,
          1.0474e-01,  1.1593e-01, -5.0180e-01, -1.8626e-01,  5.3346e-02,
          1.1946e-01, -7.9624e-02, -5.1583e-01, -5.2225e-02, -2.2669e-01,
         -9.4476e-03,  1.2027e-01,  4.4413e-01, -1.3955e-01,  1.5552e-01,
         -4.3507e-01, -1.3689e-01,  1.9468e-01, -7.1899e-02,  1.9375e-02,
         -6.3177e-02, -1.2233e-01,  5.8654e-02,  8.0059e-02,  7.4969e-02,
         -3.4419e-01, -9.6314e-02, -7.7673e-02, -3.2606e-01,  4.2313e-01,
          5.6200e-02,  6.7163e-01, -1.5191e-02,  1.5063e-01, -3.6597e-02,
          2.6365e-01,  2.4447e-01, -1.

In [11]:
cls_embedding.shape

torch.Size([1, 768])

In [12]:
testdf = df.iloc[0:3,2:5]
testdf
#testdf['embedding'] = cls_embedding
#testdf[['Patient ID', 'embedding']]

,Patient ID,Prompts,Reports
0,P0001,Generate a structured clinical note based on t...,Clinical Note\n\nPatient Demographics\n\n* Age...
1,P0002,Generate a structured clinical note based on t...,Clinical Note\n\nDemographics\n\n* Patient's N...
2,P0003,Generate a structured clinical note based on t...,Clinical Note\n\nPatient Information:\n\n* Nam...


In [15]:
# Sample DataFrame with patient IDs and clinical reports
# data = {
#     'patient_id': ['patient_1', 'patient_2', 'patient_3'],
#     'clinical_report': [
#         'Patient has a history of hypertension and diabetes.',
#         'Patient is recovering well after surgery.',
#         'Patient has chronic asthma with frequent exacerbations.'
#     ]
# }

# df = pd.DataFrame(data)
df2 = df.iloc[0:3, :]

# Function to get text embeddings
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # Get the [CLS] token embedding (first token)
    embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()  # Remove batch dimension
    return embedding

# Use tqdm to monitor the progress of applying the function to the DataFrame
tqdm.pandas(desc="Processing clinical reports")

# Apply the function with tqdm progress bar
df2['embedding'] = df2['Reports'].progress_apply(get_embedding)

# Now, create a new DataFrame with patient ID and corresponding embeddings
embedding_df = df2[['Patient ID', 'embedding']]

# View the new DataFrame
print(embedding_df)

Processing clinical reports: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.01it/s]

  Patient ID                                          embedding
0      P0001  [-0.15973704, -0.004579649, -0.412982, -0.0061...
1      P0002  [-0.13748477, 0.07512919, -0.4832943, -0.07459...
2      P0003  [-0.13879517, 0.040618695, -0.45822608, 0.1434...



/tmp/ipykernel_3352872/3164658054.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['embedding'] = df2['Reports'].progress_apply(get_embedding)


In [17]:
embedding_df['embedding'][0].shape

(768,)